## MCP — Model Context Protocol
#### A Hands-On Guide: Understand · Use · Build · Connect

[![Open In Colab](https://colab.research.google.com/github/afafelwafi/hackai-2026/blob/mcp/notebooks/01_mcp_agents_new.ipynbassets/colab-badge.svg)](https://colab.research.google.com/github/afafelwafi/hackai-2026/blob/mcp/notebooks/01_mcp_agents_new.ipynb)

| Section | What you will learn |
|---------|---------------------|
| **0 · What is MCP?** | Problem it solves, before/after, architecture, primitives |
| **1 · Setup** | Install packages, set API keys |
| **2 · Use an existing MCP server** | Connect via smolagents (Off the shelf LLM or open-source LLM) |
| **3 · Build your own MCP server** | File structure, tools/resources/prompts, launch & test |
| **4 · WhatsApp MCP + Claude Desktop** | End-to-end walkthrough with the community whatsapp-mcp |


### 0 · What is MCP?

#### 0.1 The Problem — LLMs are powerful but isolated

Once an LLM is trained, it is **frozen in time**. It knows everything up to its training cutoff,
and nothing after. But more fundamentally — it can only generate text. It cannot *do* things.

> Imagine you ask your AI assistant:
> *"Help me build a presentation on Canva about climate change in Morocco."*
>
> It can write beautiful slide content. But it cannot open Canva, create a design,
> add slides, pick a template, or export a PDF. It has no hands.
> It doesn\'t know how to *use* Canva the way a human does.


#### The first fix: Tools (function calling)

Developers found a workaround — **tools**. The idea is simple:

> Wrap a real action in a Python function, give it a clear name and description,
> and tell the LLM *"you can call this whenever you need to."*

```python
def create_canva_slide(title: str, content: str) -> str:
    # Creates a new slide in Canva with the given title and content
    response = canva_api.post("/slides", {"title": title, "content": content})
    return response["slide_url"]
```

Now the LLM can trigger real actions. You pass the function definition alongside your prompt,
the model decides when to call it, and your code executes it.

**LLM + tools = an agent that can act in the world.** ✅


#### The new problem: N × M integrations

It works — but it doesn\'t scale.

Every time you want your LLM to talk to a new app, someone has to write custom tool code for it.
And every time a new LLM app is built, it has to rewrite all those tools from scratch.

<img src="https://i.postimg.cc/bJ1TXFcW/image.png" width="700"/>

With **N AI apps** and **M external tools** you end up maintaining **N × M** bespoke connectors —
each with its own auth, data format, error handling, and versioning.
The same Canva integration gets rewritten for Claude Desktop, for Cursor, for your custom agent...

> **This is exactly as tedious as it sounds.**
> Change Canva\'s API → update N connectors.
> Add a new app → write M new tools.
> Build a new AI product → start from zero.

#### 0.2 The Fix — MCP (Model Context Protocol)

MCP is an **open standard** that answers one question:

> *What if every tool/service exposed itself in a universal format,
> and every AI app knew how to speak that format?*

Instead of N × M custom connectors, you get:
- Each service builds **one MCP server** — once.
- Each AI app implements **one MCP client** — once.
- They all talk the same language.

<img src="https://i.postimg.cc/gjDHjX0K/image.png" width="700"/>

The Canva team ships a Canva MCP server. You connect Claude Desktop, Cursor, your custom agent —
any of them — with zero additional code. Someone updates the Canva API? Only the MCP server
changes. Your agents stay untouched.

<img src="https://i.postimg.cc/yxv0BMfQ/image.png" width="700"/>

> Think of it as **REST for AI tool integration**.
> REST standardised how web apps talk to backends.
> MCP standardises how AI apps talk to tools and data sources.

| | Before MCP | With MCP |
|---|---|---|
| Integration effort | N x M | N + M |
| Portability | Zero — each connector is bespoke | Complete — any client works with any server |
| Maintenance | Every team maintains theirs | Server author ships one update, everyone benefits |

#### 0.3 MCP Architecture

MCP follows a clean **client–server** architecture.

<img src="https://i.postimg.cc/SxXcFtcd/image.png" width="700"/>

| Role | Description | Examples |
|------|-------------|---------|
| **Host** | The LLM application that *wants* external capabilities | Claude Desktop, Cursor, your agent |
| **MCP Client** | Lives inside the host; speaks the MCP wire protocol | Built into the host |
| **MCP Server** | Lightweight process that *exposes* capabilities | darija-toolkit, GitHub MCP, WhatsApp MCP |

**Transport options**
- `stdio` — host spawns the server as a child process; simplest for local use.
- `streamable-http` — server listens on an HTTP endpoint; great for remote/cloud servers.
- `SSE (Server-Sent Events)` s a one-way HTTP streaming protocol — legacy, being replaced by streamable-http.


#### 0.4 What a Server Exposes: The Three Primitives

<img src="https://i.postimg.cc/VvG02j62/image.png" width="700"/>

| Primitive | Controlled by | Purpose | Example |
|-----------|--------------|---------|---------|
| **Tools** | **Model** (AI decides when to call) | Executable functions; can have side-effects | `send_message`, `search_web`, `transliterate` |
| **Resources** | **Application** (app exposes, user picks) | Read-only, static-ish data blobs | Stopword list, a file, a knowledge base |
| **Prompts** | **User** (user selects a template) | Pre-built prompt templates/workflows | "Summarize in Darija", "Code review" |

> **Rule of thumb:** if it *does* something → Tool. If it *is* data → Resource. If it *guides* the conversation → Prompt.


### 1 · Setup

In [1]:
packages = [
    "mcp",
    "fastmcp",
    "smolagents[mcp,litellm]",
    "anthropic",
    "httpx",
    "uvicorn",
    "rich",
]

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + packages, check=True)
print("All packages installed.")


All packages installed.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os

# Hugging Face (for gated models)
os.environ.setdefault("HF_TOKEN", "")

# Google AI Studio — free tier, get key at https://aistudio.google.com
os.environ.setdefault("GEMINI_API_KEY", "")

print("Keys configured")


Keys configured


### 2 · Using an MCP Server with an LLM

We use the **darija-toolkit** (Demo MCP server for darija that can do translation, detect dialect, etc.) server from as a concrete MCP endpoint.
The client code is identical regardless of which server you point it at — local or remote.

We demonstrate an open-source LLM via the [`smolagents`](https://huggingface.co/docs/smolagents/index) (Python library for building agents) MCP client (Gemma on Google AI Studio or local Qwen).

##### 2.1 Start the darija-toolkit MCP server & Check it's alive

In [3]:
import subprocess, time, sys, os

server_path = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                           "mcp", "darija-mcp", "darija_server.py")

proc = subprocess.Popen(
    [sys.executable, server_path],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
time.sleep(3)
print(f"Server PID: {proc.pid}")
print("Endpoint:   http://127.0.0.1:8765/mcp")


Server PID: 20884
Endpoint:   http://127.0.0.1:8765/mcp


In [4]:
import subprocess, json

result = subprocess.run([
    "curl", "-s", "-X", "POST", "http://localhost:8765/mcp",
    "-H", "Content-Type: application/json",
    "-H", "Accept: application/json, text/event-stream",
    "-d", json.dumps({
        "jsonrpc": "2.0", "method": "initialize",
        "params": {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "healthcheck", "version": "0"},
        },
        "id": 1,
    }),
], capture_output=True, text=True)

for line in result.stdout.splitlines():
    if line.startswith("data:"):
        info = json.loads(line[5:])
        srv  = info["result"]["serverInfo"]
        caps = list(info["result"]["capabilities"].keys())
        print(f"Server alive ✓  name={srv['name']}  version={srv['version']}")
        print(f"Capabilities:   {caps}")


Server alive ✓  name=darija-toolkit  version=3.2.4
Capabilities:   ['experimental', 'logging', 'prompts', 'resources', 'tools', 'extensions']


##### 2.2 Open-Source LLM via `smolagents` using `darija-toolkit` MCP Server

`smolagents.MCPClient` turns every tool on the server into a native smolagents tool —
one call to `get_tools()` is the entire integration.  The agent can use any LLM backend.

In [5]:
from smolagents.mcp_client import MCPClient

mcp_client = MCPClient({
    "url": "http://127.0.0.1:8765/mcp",
    "transport": "streamable-http",
})

tools = mcp_client.get_tools()
print("Discovered tools:", [t.name for t in tools])


/tmp/ipykernel_20638/3230722733.py:3: FutureWarning: Parameter 'structured_output' was not specified. Currently it defaults to False, but in version 1.25, the default will change to True. To suppress this warning, explicitly set structured_output=True (new behavior) or structured_output=False (legacy behavior). See documentation at https://huggingface.co/docs/smolagents/tutorials/tools#structured-output-and-output-schema-support for more details.
  mcp_client = MCPClient({


Discovered tools: ['transliterate', 'detect_dialect', 'clean_arabic', 'hijri_today']


In [7]:
import os
from smolagents import CodeAgent, LiteLLMModel, TransformersModel

CLOUD_MODEL = "gemini/gemma-4-26b-a4b-it"
LOCAL_MODEL  = "Qwen/Qwen2.5-0.5B-Instruct"

def build_model():
    if os.getenv("GEMINI_API_KEY"):
        try:
            m = LiteLLMModel(
                model_id=CLOUD_MODEL,
                api_key=os.environ["GEMINI_API_KEY"],
                temperature=0.2,
                max_tokens=1024,
            )
            m([{"role": "user", "content": "ping"}])
            print("Using Google AI Studio (Gemma).")
            return m
        except Exception as e:
            print(f"Cloud unavailable: {e}")
    print("Falling back to local Qwen model.")
    return TransformersModel(model_id=LOCAL_MODEL, device_map="auto", max_new_tokens=512)

model = build_model()
agent = CodeAgent(tools=list(tools), model=model,
                  additional_authorized_imports=["json"], max_steps=10)

QUERY = """
3tini résumé dyalhad texte, w 9oli wach hia darija wlla 3arabia fos7a:
Charek f 1337 HackAI 2026, w t3alamt bzaf 3la MCP w kifach n9dar nbni servers.
"""
response = agent.run(QUERY)
print(response)

Using Google AI Studio (Gemma).


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 3tini résumé dyalhad texte, w 9oli wach hia darija wlla 3arabia fos7a:                                          │
│ Charek f 1337 HackAI 2026, w t3alamt bzaf 3la MCP w kifach n9dar nbni servers.                                  │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemma-4-26b-a4b-it ──────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = "Charek f 1337 HackAI 2026, w t3alamt bzaf 3la MCP w kifach n9dar nbni servers."                          
  dialect = detect_dialect(text=text)                                                                              
  print(f"Dialect: {dialect}")                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Dialect: {"dialect":"darija","confidence":1.0,"markers":["bzaf"]}

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Résumé: L-chakhs charek f 1337 HackAI 2026 w t3alam 3la MCP w kifach ibni servers. Dialect:        
  Darija.")                                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Résumé: L-chakhs charek f 1337 HackAI 2026 w t3alam 3la MCP w kifach ibni servers. Dialect: Darija.

[Step 2: Duration 25.62 seconds| Input tokens: 4,992 | Output tokens: 848]

Résumé: L-chakhs charek f 1337 HackAI 2026 w t3alam 3la MCP w kifach ibni servers. Dialect: Darija.


### 3 · Build Your Own MCP Server — `darija-toolkit`

Building an MCP server with **FastMCP** (a Python framework designed to build MCP servers).
It's as simple as building a FastAPI app: you define Python functions and decorate them,
FastMCP handles the wire protocol.


##### 3.1 File Structure

```
darija-mcp/
├── pyproject.toml        # package metadata & pinned dependencies
├── server.json           # MCP registry metadata (mcp.so / Smithery listing)
├── README.md             # docs for consumers of your server
└── darija_server.py      # all your logic lives here
```

Only `darija_server.py` and `pyproject.toml` are strictly required.
`server.json` is needed only if you want to publish to the MCP registry.


#### 3.2 `pyproject.toml` — Package Metadata

```toml
[project]
name = "darija-mcp"
version = "0.1.3"
description = "MCP server with Darija NLP tools"
requires-python = ">=3.10"
dependencies = ["fastmcp>=2.5", "mcp>=1.4"]

[project.scripts]
darija-mcp = "darija_mcp.server:main"   # CLI entry-point (optional)
```

Standard Python packaging file. Declares your server's name, version, and dependencies.
`fastmcp` + `mcp` are the only mandatory deps — everything else is your business logic.
The `[project.scripts]` entry creates a CLI command so users can run `darija-mcp start` instead of `python darija_server.py`.


##### 3.3 `server.json` — MCP Registry Metadata *(optional)*

```json
{
  "$schema": "https://static.modelcontextprotocol.io/schemas/2025-09-29/server.schema.json",
  "name": "io.github.yourhandle/darija-mcp",
  "description": "Darija NLP utilities",
  "version": "0.1.3",
  "repository": { "url": "https://github.com/yourusername/darija-mcp", "source": "github" },
  "packages": [
    { "registryType": "pypi", "identifier": "darija-mcp", "version": "0.1.3",
      "transport": { "type": "stdio" } }
  ]
}
```

Parsed by [mcp.so](https://mcp.so) and Smithery so your server appears in the public registry
and users can install it with one click. The `transport` field tells the registry how to
launch your server (`stdio` for local, `streamable-http` for remote).

| Transport | When to use |
|-----------|-------------|
| `stdio` | Claude Desktop, Cursor, any app that spawns the server as a child process |
| `streamable-http` | Notebook demos, cloud deployments, any remote endpoint |


##### 3.4 `darija_server.py` — The Server Itself

The single file where all your MCP logic lives. It contains three types of primitives
(Tools, Resources, Prompts) — each covered in detail below.

In [9]:
# Display the server source with syntax highlighting
from rich.syntax import Syntax
from rich.console import Console
import pathlib

src = pathlib.Path("mcp/darija-mcp/darija_server.py").read_text()
Console().print(Syntax(src, "python", line_numbers=True, theme="monokai"))


   1 from fastmcp import FastMCP                                                                                   
   2 from datetime import datetime                                                                                 
   3 import re, unicodedata, json, urllib.request                                                                  
   4                                                                                                               
   5 mcp = FastMCP("darija-toolkit")                                                                               
   6                                                                                                               
   7 # TOOLS                                                                                                       
   8 @mcp.tool                                                                                                     
   9 def transliterate(text: str) -> str:                                                                          
  10     """Convert Arabic script to Latin (Buckwalter-ish). Useful for logs."""                                   
  11     table = str.maketrans({                                                                                   
  12         "ا":"a","ب":"b","ت":"t","ث":"th","ج":"j","ح":"7","خ":"kh","د":"d",                                    
  13         "ذ":"dh","ر":"r","ز":"z","س":"s","ش":"sh","ص":"S","ض":"D","ط":"T",                                    
  14         "ظ":"Z","ع":"3","غ":"gh","ف":"f","ق":"q","ك":"k","ل":"l","م":"m",                                     
  15         "ن":"n","ه":"h","و":"w","ي":"y","ى":"a","ة":"a","ء":"a",                                              
  16     })                                                                                                        
  17     return unicodedata.normalize("NFKC", text)[38;2;25

#### Primitive 1: Tools  *(model-controlled)*

```python
from fastmcp import FastMCP

mcp = FastMCP("darija-toolkit")   # server name shown in the host's UI

@mcp.tool
def transliterate(text: str) -> str:
    """Convert Arabic script to Latin (Buckwalter-ish). Useful for logs."""
    ...
```

**Rules:**
- Decorate with `@mcp.tool`.
- The **docstring** becomes the tool description the LLM reads — write it clearly.
- Type-annotated parameters auto-generate the JSON Schema the model fills in.
- Return type can be `str`, `dict`, `list` — FastMCP serialises it.
- Tools **can** have side-effects (write to DB, send an email, call an API).


#### Primitive 2: Resources  *(application-controlled)*

```python
@mcp.resource("darija://stopwords")
def stopwords() -> str:
    """The 1337AI Darija stopword list."""
    sw = ["bach", "ghir", "ghadi", "kayn", ...]
    return "\n".join(sw)
```

**Rules:**
- Decorate with `@mcp.resource("scheme://path")`.
- Must be **side-effect free** — think of it as a GET endpoint.
- Use resources for static-ish data: lists, configs, knowledge bases.


#### Primitive 3: Prompts  *(user-controlled)*

```python
@mcp.prompt
def summarize_in_darija(text: str) -> str:
    """Summarize the given text in 3 sentences of Darija."""
    return f"Lakhas had nass w 3tini résumé f tlata jamals b darija:\n\n{text}"
```

**Rules:**
- Decorate with `@mcp.prompt`.
- Returns a **string** (the prompt text) that the host injects into the conversation.
- Parameters let the user customise the template at runtime.
- Prompts appear in the host UI as slash-command-like shortcuts.


#### 3.5 Transport & Launch

The last line of your server file picks the transport:

```python
if __name__ == "__main__":
    # For Claude Desktop / local host (stdio)
    mcp.run(transport="stdio")

    # For remote / notebook use (HTTP) 
    mcp.run(transport="streamable-http", host="0.0.0.0", port=8765)
```

### 4 · Practical Use Case: using claude desktop with a WhatsApp MCP Server (Better to run this section on a Local machine)

#### 4.1 Architecture

The whatsapp-mcp project is split into two processes that talk over a local HTTP socket:

```
Claude Desktop
     │  (stdio MCP)
     ▼
whatsapp-mcp-server/   ← Python MCP server (the MCP layer)
     │  (HTTP :8080)
     ▼
whatsapp-bridge/       ← Go application (the WhatsApp layer)
     │  (WhatsApp Web multidevice API / whatsmeow)
     ▼
WhatsApp servers
```

**Data privacy:** all messages are stored in a local SQLite database inside
`whatsapp-bridge/store/`.  Only what the agent explicitly requests is sent to Claude.

#### 4.2 Available MCP Tools

Once connected, Claude can call:

| Tool | What it does |
|------|-------------|
| `search_contacts` | Find contacts by name or phone number |
| `list_chats` | List chats with metadata |
| `list_messages` | Retrieve messages with filters |
| `get_last_interaction` | Most recent message with a contact |
| `send_message` | Send a text message |
| `send_file` | Send image / video / document |
| `send_audio_message` | Send a voice note (.ogg Opus) |
| `download_media` | Download media from a message |

#### 4.3 Prerequisites

Before you start, make sure you have:

- [ ] **Claude Desktop** installed ([Mac/Windows](https://claude.ai/download) · ([Ubuntu](https://github.com/aaddrick/claude-desktop-debian)))
- [ ] **Go ≥ 1.21** — `go version`
- [ ] **UV** — `curl -LsSf https://astral.sh/uv/install.sh | sh`
- [ ] **Python 3.6+** — already satisfied if you are in this notebook
- [ ] *(optional)* **FFmpeg** — only needed to auto-convert audio to .ogg Opus voice notes

#### Step 1 — Locate the local WhatsApp MCP

In [3]:
import subprocess, sys, os, pathlib, platform

WHATSAPP_MCP = pathlib.Path("mcp/whatsapp-mcp").resolve()
BRIDGE_DIR   = WHATSAPP_MCP / "whatsapp-bridge"
SERVER_DIR   = WHATSAPP_MCP / "whatsapp-mcp-server"

assert BRIDGE_DIR.exists(), f"Bridge not found at {BRIDGE_DIR}"
assert SERVER_DIR.exists(), f"Server not found at {SERVER_DIR}"
print("Local whatsapp-mcp found")


# Install Go if not present
def install_go():
    result = subprocess.run(["go", "version"], capture_output=True, text=True)

    if result.returncode == 0:
        print(f"Go already installed: {result.stdout.strip()}")
    elif platform.system() == "Linux":
        subprocess.run([
            "bash", "-c",
            "wget -q https://go.dev/dl/go1.24.3.linux-amd64.tar.gz && "
            "sudo tar -C /usr/local -xzf go1.24.3.linux-amd64.tar.gz"
        ], check=True)
        os.environ["PATH"] += ":/usr/local/go/bin"
        print("Go installed.")
    else:
        sys.exit(
            f"Go not found on {platform.system()}.\n"
            "Mac    → brew install go\n"
            "Windows→ https://go.dev/dl/  (download the .msi installer)"
        )

install_go()

Local whatsapp-mcp found
Go already installed: go version go1.22.2 linux/amd64


#### Step 2 — Build & run the WhatsApp bridge (shows QR in notebook).
Scan the QR with your phone's WhatsApp (Settings → Linked Devices → Link a Device).

In [ ]:
import subprocess, threading, os, pathlib, platform

# Only Windows needs CGO enabled — Mac and Linux have it on by default
env = {**os.environ, "CGO_ENABLED": "1"} if platform.system() == "Windows" else os.environ

print("Building bridge (first time ~30s)...")
subprocess.run(["go", "build", "-o", "bridge", "."], cwd=BRIDGE_DIR, env=env, check=True)
print("Build done.\n")

bridge_proc = subprocess.Popen(
    ["./bridge"], cwd=BRIDGE_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

def stream():
    for line in bridge_proc.stdout:
        print(line, end="")

threading.Thread(target=stream, daemon=True).start()
print("Scan the QR code above: WhatsApp → Settings → Linked Devices → Link a Device")

# You will see your messages in the notebook output as you interact with the linked WhatsApp account.


Building bridge (first time ~30s)...
Build done.

Scan the QR code above: WhatsApp → Settings → Linked Devices → Link a Device


11:21:24.123 [Client INFO] Starting WhatsApp client...
11:21:25.043 [Client INFO] Successfully authenticated
11:21:25.623 [Client INFO] Connected to WhatsApp

✓ Connected to WhatsApp! Type 'help' for commands.


### Step 3 — Configure Claude Desktop

In [ ]:
import json, shutil, pathlib, platform

uv_path    = shutil.which("uv") or "/usr/local/bin/uv"
server_dir = str(pathlib.Path("mcp/whatsapp-mcp/whatsapp-mcp-server").resolve())

# locate the config file
config_path = {
    "Darwin":  pathlib.Path.home() / "Library/Application Support/Claude/claude_desktop_config.json",
    "Windows": pathlib.Path.home() / "AppData/Roaming/Claude/claude_desktop_config.json",
    "Linux":   pathlib.Path.home() / ".config/Claude/claude_desktop_config.json",
}.get(platform.system())


# read existing config 
existing = json.loads(config_path.read_text()) if config_path.exists() else {}
existing.setdefault("mcpServers", {})

# add / overwrite the whatsapp entry
existing["mcpServers"]["whatsapp"] = {
    "command": uv_path,
    "args": ["--directory", server_dir, "run", "main.py"]
}

config_path.write_text(json.dumps(existing, indent=2))
print(f"Saved to: {config_path}")
print("\nCurrent mcpServers:", list(existing["mcpServers"].keys()))
print("\nRestart Claude Desktop to apply changes.")

# You will see your whastapp messages appear in the cosole when you interact with the linked WhatsApp account.
# Outut will look like:
# [2024-11-05 12:34:56] Received message from +1234567890: "Hello from WhatsApp!"

### Step 4 — Restart Claude Desktop & Verify

1. **Restart** Claude Desktop completely (Quit, not just close the window).
2. Open a new conversation. You can see the WhatsApp MCP server and its tools in the tool picker. <br>
 <img src="https://i.postimg.cc/gkX1XC0t/Screenshot-from-2026-05-15-11-17-22.png" width="700"/> 
 <br>
3. Try: *"List my 5 most recent WhatsApp chats"*

If the server does not appear, check:
- The Go bridge is still running in its terminal.
- The paths in `claude_desktop_config.json` are absolute and correct.
- Logs: `~/Library/Logs/Claude/mcp-server-whatsapp.log` (macOS).


### ⚠️ Security Notice

The WhatsApp MCP server — like most MCP servers with read/write access to personal data —
is subject to **prompt injection**:
a malicious message in your WhatsApp inbox could instruct Claude to exfiltrate other
messages or send messages on your behalf.

**Mitigations:**
- Only run the server when you actively need it (do not leave it always-on).
- Review what Claude is about to do before confirming sensitive operations.
- Keep `whatsapp-bridge/store/` outside any cloud-synced folder.


#### 🔥🔥🔥 **Can you think of a custom, Morocco ecosystem–aware MCP server you would build? What tools, resources, or prompts would it include? How could you make it public so others can use it? Share your ideas in the community channels!**